# **Finale Quizzz of NLP**

> by Michael Ahlovely Stevenson

In [1]:
import pickle
import pandas as pd
import numpy as np
import spacy
import os
from string import punctuation
import random

from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, SnowballStemmer
from nltk.tag import pos_tag
from nltk.probability import FreqDist
from nltk.classify import NaiveBayesClassifier, accuracy

In [2]:
# import nltk
# nltk.download('punkt')
# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('averaged_perceptron_tagger')
# exit()

### **Utility**

In [3]:
stemmerx = SnowballStemmer('english')
lemmatizerx = WordNetLemmatizer()
stopwordx = stopwords.words('english')
categories = {}
data = pd.read_csv('./jobpostingdata.csv')

print(data.isnull().sum())
data = data.dropna()

X = data['title'] + ' ' + data['text']
Y = data['fraudulent']

title         0
fraudulent    0
text          0
dtype: int64


### **Preprocessing**

In [4]:
def AlterTag (tag: str):
    if tag.startswith('J'):
        return 'a'
    elif tag.startswith('V'):
        return 'v'
    elif tag.startswith('R'):
        return 'r'
    return 'n'
    
def Preprocessing (docx: str):
    tokens = word_tokenize(docx.lower())
    tokens = [tok for tok in tokens if tok not in punctuation]
    tokens = [tok for tok in tokens if tok.isalpha()]
    tokens = [tok for tok in tokens if tok not in stopwordx]

    tagged = pos_tag(tokens)
    # tokens = [stemmerx.stem(tok) for tok in tokens]
    tokens = [lemmatizerx.lemmatize(tok, AlterTag(tag)) for tok, tag in tagged]

    return tokens


### **Training**

In [5]:
data = pd.read_csv('./jobpostingdata.csv')
data.head()

,title,fraudulent,text
0,PHP Developer,0,PHP Developer You're a skilled developer. You ...
1,CUSTOMER SERVICE AGENT,1,CUSTOMER SERVICE AGENT Aegis is a global busi...
2,VP Marketing & Growth,0,VP Marketing & Growth Depop is an exciting new...
3,SAP BW Developer/Architect,1,SAP BW Developer/Architect Assist with Full L...
4,Administrative Assistant,1,Administrative Assistant With decades of exper...


In [6]:
def Training ():
    # Feature Extraction
    feats = []

    all_tokens = Preprocessing(' '.join(X))
    freqs = FreqDist(all_tokens)
    print(freqs.most_common(5))

    for text, label in zip(X, Y):
        clean = Preprocessing(text)
        feat = {word: True for word in clean}
        feats.append((feat, label))
    
    random.shuffle(feats)

    # Training
    split = int(len(feats) * 0.8)
    train_data = feats[:split]
    evals_data = feats[split:]
    print('Start Training...')

    model = NaiveBayesClassifier.train(train_data)
    acc = accuracy(model, evals_data)

    print('Model Trained')
    print(f'Accuracy: {acc*100}%')
        
    # Info
    print('Top 5 Most Informative Features:')
    print(model.most_informative_features(5))

    # Save
    with open('./model.pickle', 'wb') as file:
        pickle.dump(model, file)
    print('Model Saved')

    return model

def Load ():
    if os.path.exists('./model.pickle'):
        with open('./model.pickle', 'rb') as file:
            model = pickle.load(file)
        return model
    else:
        print("No Model Detected!")
        model = Training()
        return model

### **Embedding Models**

In [7]:
def TF_IDF (query):
    vectorizer = TfidfVectorizer(stop_words='english')
    matrix = vectorizer.fit_transform(X)
    qvector = vectorizer.transform([query])

    similarity = cosine_similarity(qvector, matrix).flatten()
    data['Similarity'] = similarity

    sorted_data = data.sort_values(by='Similarity', ascending=False)

    print('Top 5 Job Recommendation for You')
    print(f'1. :{sorted_data.iloc[0,0]}')
    print(f'2. :{sorted_data.iloc[1,0]}')
    print(f'3. :{sorted_data.iloc[2,0]}')
    print(f'4. :{sorted_data.iloc[3,0]}')
    print(f'5. :{sorted_data.iloc[4,0]}')

def NGrams (query):
    vectorizer = TfidfVectorizer(ngram_range=(1, 3), stop_words='english')
    matrix = vectorizer.fit_transform(X)
    qvector = vectorizer.transform([query])

    similarity = cosine_similarity(matrix, qvector).flatten()
    data['Similarity'] = similarity

    sorted_data = data.sort_values(by='Similarity', ascending=False)
    print('Top 5 Job Recommendation for You')
    print(f'1. :{sorted_data.iloc[0,0]}')
    print(f'2. :{sorted_data.iloc[1,0]}')
    print(f'3. :{sorted_data.iloc[2,0]}')
    print(f'4. :{sorted_data.iloc[3,0]}')
    print(f'5. :{sorted_data.iloc[4,0]}')

### **NER**

In [ ]:
def StartNER():
    paragraph = ' '.join(data['text'].head(100))
    # print(len(word_tokenize(paragraph)))

    ner = spacy.load('en_core_web_sm')
    # ner.max_length = 10000000
    paragraph = ner(paragraph)

    for ent in paragraph.ents:
        label = ent.label_
        if label not in categories:
            categories[label] = []
        categories[label].append(ent.text)

### **Support Function**

In [9]:
docx = 'No Text'
categorex = 'No Category'

In [10]:
def Enter():
    input('Press [Enter] to continue...')
    print()
    print()
    print()

def WriteText(model):
    global docx, categorex

    while True:
        print('Please Input your Text: ')
        docx = input('')

        if len(docx.split()) < 20:
            print('(!) Please input 20 words or more')
        else:
            tokens = Preprocessing(docx)
            feats = {word: True for word in tokens}
            categorex = model.classify(feats)
            print(f'Your Text Category: {categorex}')
            break

def ViewRecomm ():
    if (docx == 'No Text'):
        print('Please Input your Text First')
        return
    
    print('Choose Language Model: ')
    print('1. TF-IDF')
    print('2. N-Grams')
    cc = input('')

    if cc == '1':
        TF_IDF(docx)
    elif cc == '2':
        NGrams(docx)
    else:
        print('Invalid Input')

def ViewNER ():
    if not categories:
        StartNER()
    
    for label, ent in categories.items():
        print(f'{label}: {", ".join(ent)}')

### **Main Menu**

In [11]:
def Main ():
    print('Loading Model...')
    model = Load()
    
    while True:
        print('')
        print('Ril or Fek')
        print('Di PDF namanya gitu :)')
        print(f'Your Text: {docx}')
        print(f'Your Text Category: {categorex}')
        print('================')
        print('1. Write Your Text')
        print('2. View Job Recommendation')
        print('3. View Named Entity Recognizer')
        print('4. Exit')   
        print('>> ')
        cc = input('')
        print('')

        if cc == '1':
            WriteText(model)
        elif cc == '2':
            ViewRecomm()
        elif cc == '3':
            ViewNER()
        elif cc == '4':
            break
        else:
            print('Invalid Input')

In [12]:
Main()

Loading Model...

Ril or Fek
Di PDF namanya gitu :)
Your Text: No Text
Your Text Category: No Category
1. Write Your Text
2. View Job Recommendation
3. View Named Entity Recognizer
4. Exit
>> 

Please Input your Text: 
Your Text Category: 1

Ril or Fek
Di PDF namanya gitu :)
Your Text: software engineer bla bla bla good bad trump prabowo software engineer bla bla bla good bad trump prabowo software engineer bla bla bla good bad trump prabowo
Your Text Category: 1
1. Write Your Text
2. View Job Recommendation
3. View Named Entity Recognizer
4. Exit
>> 

Choose Language Model: 
1. TF-IDF
2. N-Grams
Top 5 Job Recommendation for You
1. :Senior Software Engineer QA Automation
2. :Automation Software Engineer
3. :Software Engineer | Web Development
4. :Software Engineer, iOS
5. :Software Engineer | Forecasting & Optimization

Ril or Fek
Di PDF namanya gitu :)
Your Text: software engineer bla bla bla good bad trump prabowo software engineer bla bla bla good bad trump prabowo software engineer